In [ ]:
# Cell 1: Verify GPU
import subprocess
import torch

subprocess.run(['nvidia-smi'], check=True)
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Device         : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# Cell 2: Clone GitHub repo
import os

REPO_URL  = 'https://github.com/krantiprakash/VisDrone2019-Multi-Object-Detection-and-Tracking.git'
REPO_NAME = 'VisDrone2019-Multi-Object-Detection-and-Tracking'
WORK_DIR  = f'/kaggle/working/{REPO_NAME}'

if not os.path.exists(WORK_DIR):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, WORK_DIR],
        capture_output=True, text=True
    )
    print(result.stdout)
    print(result.stderr)
else:
    result = subprocess.run(
        ['git', '-C', WORK_DIR, 'pull'],
        capture_output=True, text=True
    )
    print(result.stdout)

os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

In [ ]:
# Cell 3: Install dependencies
import subprocess

subprocess.run(['pip', 'install', '-r', 'requirements.txt', '-q'], check=True)
subprocess.run(
    ['pip', 'install',
     'git+https://github.com/KaiyangZhou/deep-person-reid.git', '-q'],
    check=True
)
subprocess.run(
    ['pip', 'install',
     'git+https://github.com/JonathonLuiten/TrackEval.git', '-q'],
    check=True
)
print('All dependencies installed.')

In [ ]:
# Cell 4: Verify dataset paths
import sys
sys.path.append(WORK_DIR)

from configs.paths import get_paths, verify_paths
paths = get_paths()
verify_paths(paths)
print('\nResolved paths:')
for key, val in paths.items():
    print(f'  {key:<20} : {val}')

In [ ]:
# Cell 5: Convert VisDrone annotations to YOLO format
result = subprocess.run(
    ['python', 'src/data/convert_to_yolo.py'],
    capture_output=False,
    text=True
)
if result.returncode != 0:
    raise RuntimeError('convert_to_yolo.py failed. Check output above.')
print('Conversion complete.')

In [ ]:
# Cell 6: Fine-tune YOLO26x on VisDrone-DET
result = subprocess.run(
    ['python', 'src/detection/train_yolo.py'],
    capture_output=False,
    text=True
)
if result.returncode != 0:
    raise RuntimeError('train_yolo.py failed. Check output above.')

In [ ]:
# Cell 7: Verify outputs available for download
from pathlib import Path

download_dir = paths['out_detection'] / 'download'

print('Files available for download:')
if download_dir.exists():
    for f in sorted(download_dir.iterdir()):
        size_mb = f.stat().st_size / 1e6
        print(f'  {f.name:<40} {size_mb:.1f} MB')
else:
    print(f'  Download folder not found: {download_dir}')